# Расчёт размера выборки для A/B теста

В нашем проекте тест длился 1 неделю. И количество наблюдений (количество сессий) не изменить. 

Рассчитаем: 
- Какой размер выборки был необходим для проведения АВ-теста при получившемся потоке трафика в день, если нам нужно было замерить изменение только для paid CTR?
- Какая нужна длительность теста, чтобы увидеть __*различия в 5%*__ в метрике конверсия в клик по платному объявлению (paid CTR )? Базовый __*paid CTR = 2%*__. 

In [ ]:
# загрузим датасет
import pandas as pd

df = pd.read_excel('marketpele_ab_test.xlsx')

Определим минимальное количество сессий в день для одной группы.

In [55]:
# посчитаем количество сессий в день по группам
df_daily = df.groupby(['date','group_name'])['sessions'].sum().reset_index()

min_obs = min([df_daily[df_daily.group_name=='A']['sessions'].min(), df_daily[df_daily.group_name=='B']['sessions'].min()])

print(f'Минимальное количество сессий в день для одной группы {min_obs}')

Минимальное количество сессий в день для одной группы 59250


In [57]:

import statsmodels.stats.api as sms
from statsmodels.stats.power import TTestIndPower

# 1. Параметры
p1 = 0.02  # Базовый paid CTR = 2%
mde = 0.05 # минимально детектируемый эффект 5% - в метрике конверсия в клик по платному объявлению (paid CTR )
p2 = p1 * (1 + mde)  # Ожидаемая конверсия (p1 + MDE)
alpha = 0.05  # Уровень значимости 5%
power = 0.8  # Мощность теста 80%

print(f'Базовый CTR = {p1*100}%, Ожидемый CTR = {p2*100}%, Минимальный % различий (MDE) = {mde*100}%, Уровень значимости = {alpha*100}%, Мощность = {power*100}%')

# 2. Расчет эффекта Коена 
effect_size = sms.proportion_effectsize(p1, p2)

# 3. Расчет размера выборки
analysis = TTestIndPower()
sample_size = analysis.solve_power(
    effect_size=effect_size, 
    power=power,
    nobs1=None, # The number of observations of sample two is ratio times the size of sample 1, i.e. nobs2 = nobs1 * ratio
    ratio=1.0, # предполагаем размер выборок равным - ratio of the number of observations in sample 2 relative to sample 1
    alpha=alpha
    )
print(f'Необходимый размер выборки (сессий) для одной группы: {round(sample_size) },\nТребуемый общий размер наблюдений не менее {2*round(sample_size)}.')

print(f'Если количество сессий в день для одной группы {min_obs}, то минимальная продолжительность эксперимента составит {-(-round(sample_size/min_obs,1)//1)} дней.')


Базовый CTR = 2.0%, Ожидемый CTR = 2.1%, Минимальный % различий (MDE) = 5.0%, Уровень значимости = 5.0%, Мощность = 80.0%
Необходимый размер выборки (сессий) для одной группы: 315161,
Требуемый общий размер наблюдений не менее 630322.
Если количество сессий в день для одной группы 59250, то минимальная продолжительность эксперимента составит 6.0 дней.


## Определим размер выборки с использованием калькулятора Эвана Миллера

![EvanMiller_sample_size_calc.jpg](EvanMiller_sample_size_calc.jpg)

Получили размер наблюдений для одной выборки 309928, тогда время проведения эксперимента составит 6 дней.

## Определим размер выборки с использованием калькулятора VWO

![VWO_sample_size_calc.jpg](VWO_sample_size_calc.jpg)

Получили общий размер наблюдений 775892 и  время проведения эксперимента порядка 7 дней.


__Все 3 варианта расчёта размера выборки и длительности эксперимента показали схожие результаты.__

